In [ ]:
# imports

import pandas as pd
import plotly.express as px
import plotly.io as pio

# open figures in a full browser tab instead of the cramped notebook output
# pane; switch back to "vscode" to render inline again
pio.renderers.default = "browser"

In [ ]:
from pathlib import Path

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    # Listener name is the first word of the file name,
    # e.g. "Logan Exported Data.csv" -> "Logan"
    df["Listener"] = path.stem.split()[0]
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()

### Data Cleaning

In [ ]:
# identify missing values in each column
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# handle the 1 missing values row first

# display the row with missing values - NOT THE GENRES MISSING VALUES
data[data.isnull().any(axis=1) & data['Mode'].isnull()]

In [ ]:
# drop track uri spotify:episode:3bX59hUSv3Lh3vBxmmmcNP due to bad data in the row
data = data[data['Track URI'] != 'spotify:episode:3bX59hUSv3Lh3vBxmmmcNP']

In [ ]:
# rerun missing values check

missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# drop added by column as this is an unneeded column for our analysis
data = data.drop(columns=['Added By'])

In [ ]:
# rerun missing values check
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# inspect missing values in the genres column
data[data['Genres'].isnull()]

# add a flag for missing genres
data['Missing Genres'] = data['Genres'].isnull()


def to_list(value):
    """Split a comma-separated cell into a clean list; missing becomes []."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]


# Artist Name(s) and Genres each hold several values per track, so store them
# as lists, and keep a text version for hover labels and display
for column in ["Artist Name(s)", "Genres"]:
    data[column] = data[column].apply(to_list)
    data[column + " Text"] = data[column].apply(", ".join)

data.head()

In [ ]:
data.groupby("Listener").size()

In [ ]:
data.columns.tolist()

In [ ]:
data.shape

In [ ]:
data.groupby("Listener")["Missing Genres"].sum()

In [ ]:
(
    data.explode("Genres")
    .groupby("Listener")["Genres"]
    .value_counts(dropna=False)
    .groupby(level=0)
    .head(20)
)

In [ ]:
features = [
    "Valence",
    "Energy",
    "Danceability",
    "Acousticness",
    "Tempo",
    "Loudness",
    "Popularity"
]

data.groupby("Listener")[features].describe()

In [ ]:
data.head()

In [ ]:
fig = px.scatter(
    data,
    x="Valence",
    y="Energy",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Valence vs Energy of Songs"
)

fig.show()

The graph shows that songs with higher valence often also have higher energy, but there is still a lot of variation. This suggests that valence and energy are somewhat related, but they are measuring different characteristics of a song.

In [ ]:
data.groupby("Listener")[features].corr()

The results show that energy and loudness have the strongest relationship, so songs with more energy also tend to be louder. Energy also decreases as acousticness increases. Valence and energy have a positive but moderate relationship, which supports what we saw in the scatterplot. Popularity does not seem to have a strong relationship with the other audio features.

In [ ]:
fig = px.scatter(
    data,
    x="Energy",
    y="Loudness",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Energy vs Loudness of Songs"
)

fig.show()

I made this graph because the correlation results showed that energy and loudness had the strongest relationship in our dataset. The graph makes that relationship easier to see: as songs become more energetic, they generally also become louder. This gives us another useful pair of features to consider when comparing songs.

In [ ]:
# One row per track-genre pair, dropping tracks with no genres
genre_df = data[~data["Missing Genres"]].explode("Genres").copy()
genre_df = genre_df.rename(columns={"Genres": "Genre"})

# Top 5 genres for each listener separately
top_genres = (
    genre_df.groupby("Listener")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)
top_genres

In [ ]:
# Keep only each listener's own top 5 genres
top_pairs = set(top_genres.index)

genre_summary = (
    genre_df[
        [
            pair in top_pairs
            for pair in zip(genre_df["Listener"], genre_df["Genre"])
        ]
    ]
    .groupby(["Listener", "Genre"])[["Energy", "Valence"]]
    .mean()
    .reset_index()
)

genre_summary

In [ ]:
fig = px.bar(
    genre_summary,
    x="Genre",
    y=["Energy", "Valence"],
    barmode="group",
    facet_col="Listener",
    title="Average Energy and Valence by Top Genres"
)

fig.update_xaxes(matches=None)
fig.show()

The graph shows that the top genres have different average energy and valence levels. Hip hop has the highest average energy and valence, while R&B has the lowest average energy. Rap, melodic rap, and rage rap are closer together, but still show some differences. This suggests that audio features like energy and valence may help us compare genres and could be useful later when looking for possible subgenres.

### Identifying Similarities and Differences within Genres

In [ ]:
# Lets display the total distribution of genres across all listeners, not just the top 5 for each listener. Using a barplot

# genre belongs to the music, not the listener, so count a song once even
# when both listeners liked it
unique_tracks = data.drop_duplicates(subset="Track URI")

# one row per track-genre pair, dropping tracks with no genre label
all_genres = unique_tracks[~unique_tracks["Missing Genres"]].explode("Genres")
all_genres = all_genres.rename(columns={"Genres": "Genre"})

# count how many songs fall under each genre
genre_totals = all_genres["Genre"].value_counts().reset_index()
genre_totals.columns = ["Genre", "Songs"]

# there is a long tail of rare genres, so chart the 20 biggest
top_20 = genre_totals.head(20)

# genre names are long, so put them on the y axis where they have room to
# read left to right instead of being squeezed under the bars
fig = px.bar(
    top_20,
    x="Songs",
    y="Genre",
    orientation="h",
    title="Total Songs per Genre Across All Listeners",
)

# value_counts sorts biggest first, which plots bottom up, so flip the axis
# to put the biggest genre at the top
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=600)

fig.show()

## Finding: Subgenres are often nested within the set, we need to identify common pairings to see if any genres are truly JUST a subgenre.

In [ ]:
# develop a measure of genre co-occurance. Loop through genre pairs and count how many times they appear together out of how many times the appear individually. If one appears together 100% of the time it appears individiually, then they are strongly associated (OR one is a true subgenre). Output should be a table, genre on the left, co-occurance rate top to bottom, and top 3-5 co occuring genres.

# reuse the exploded pairs from above, but a track can list the same genre
# twice when several of its artists share it, so keep one row per pair
genre_pairs = all_genres[["Track URI", "Genre"]].drop_duplicates()

# a genre needs enough songs before its rate means anything
MIN_TRACKS = 20
genre_sizes = genre_pairs["Genre"].value_counts()
common_genres = genre_sizes[genre_sizes >= MIN_TRACKS].index
genre_pairs = genre_pairs[genre_pairs["Genre"].isin(common_genres)]

# the set of songs under each genre, so overlap is just a set intersection
songs_by_genre = genre_pairs.groupby("Genre")["Track URI"].apply(set)

# for each genre, what share of ITS OWN songs also carry each other genre
rows = []
for genre, songs in songs_by_genre.items():
    for other_genre, other_songs in songs_by_genre.items():
        if other_genre == genre:
            continue

        shared = len(songs & other_songs)
        if shared == 0:
            continue

        rows.append({
            "Genre": genre,
            "Co-occurring Genre": other_genre,
            "Genre Songs": len(songs),
            "Shared Songs": shared,
            # 1.00 means every song in Genre also carries the other tag,
            # which is what a true subgenre looks like
            "Co-occurrence Rate": shared / len(songs),
        })

co_occurrence = pd.DataFrame(rows)

# the rate is directional: rap -> melodic rap is not the same as the
# reverse, and that gap is what tells child apart from parent
co_occurrence = co_occurrence.sort_values(
    ["Co-occurrence Rate", "Genre"], ascending=[False, True]
)

# keep each genre's 5 strongest partners
top_co_occurrence = co_occurrence.groupby("Genre").head(5)

top_co_occurrence.round(2)

In [ ]:
# lets maintain a seperate subgenre column for anything we know is a true subgenre (100% co occurance) which is a list of subgenre:parent genre keys.

# a true subgenre: every single one of its songs also carries the other tag
nested = co_occurrence[co_occurrence["Co-occurrence Rate"] == 1.0].copy()

# the parent has to be the broader label, so it needs more songs than the
# child; equal sizes would just mean two names for the same set
nested["Parent Songs"] = nested["Co-occurring Genre"].map(genre_sizes)
nested = nested[nested["Parent Songs"] > nested["Genre Songs"]]

# a subgenre can sit fully inside several parents, so keep the broadest one
nested = nested.sort_values("Parent Songs", ascending=False)
broadest_parent = nested.drop_duplicates(subset="Genre")

# the lookup: subgenre -> parent genre
SUBGENRE_PARENTS = dict(
    zip(broadest_parent["Genre"], broadest_parent["Co-occurring Genre"])
)

# tag every track-genre row, leaving the parent blank for top level labels
genre_pairs["Parent Genre"] = genre_pairs["Genre"].map(SUBGENRE_PARENTS)
genre_pairs["Is Subgenre"] = genre_pairs["Parent Genre"].notna()

# a grouped label that rolls subgenres up but leaves everything else alone,
# so we can compare genres at either level later
genre_pairs["Genre Grouped"] = genre_pairs["Parent Genre"].fillna(
    genre_pairs["Genre"]
)

print(f"{len(SUBGENRE_PARENTS)} true subgenres found")
SUBGENRE_PARENTS

In [ ]:
# For each measurable, I want to see every track color coded by the genre. multicode if a track is in multiple genres. ignore subgenres.

# drop the subgenre rows so a track is not plotted twice under both rap and
# melodic rap; a track still shows up once per genuinely different genre
from IPython.display import HTML, display

top_level = genre_pairs[~genre_pairs["Is Subgenre"]][["Track URI", "Genre"]]

# attach the audio features to each track-genre pair
track_features = unique_tracks[["Track URI", "Track Name"] + features]
genre_tracks = top_level.merge(track_features, on="Track URI")

# the features live in separate columns, but faceting needs them stacked
# into one value column with the feature name alongside
long_features = genre_tracks.melt(
    id_vars=["Track URI", "Track Name", "Genre"],
    value_vars=features,
    var_name="Feature",
    value_name="Value",
)

# squeeze the facets two per row instead of all seven side by side, so each
# panel is wide enough to read
fig = px.violin(
    long_features,
    x="Value",
    y="Genre",
    color="Genre",
    facet_col="Feature",
    facet_col_wrap=2,
    points=False,
    orientation="h",
    hover_data=["Track Name"],
    title="Audio Feature Distributions by Genre",
)

# tempo, loudness and popularity are on completely different scales than
# the 0-1 features, so each panel needs its own x axis
fig.update_xaxes(matches=None, showticklabels=True, title=None)

# the facet labels already name the feature, so drop the "Feature=" prefix
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# the genre names repeat down every panel, so label them once on the left
fig.update_yaxes(title=None)

# every genre needs its own band of vertical space or the violins flatten
# into each other, so size the figure off the genre count instead of
# forcing it into a fixed height
n_genres = long_features["Genre"].nunique()
n_rows = -(-len(features) // 2)
fig.update_layout(
    height=n_rows * max(300, n_genres * 28),
    showlegend=False,
)

# that figure is far taller than the screen, so put it in a fixed-height
# box of its own that scrolls instead of swamping the notebook

display(HTML(
    '<div style="height: 700px; overflow-y: scroll;">'
    + fig.to_html(include_plotlyjs="cdn", full_html=False)
    + "</div>"
))